In [0]:

# Notebook: 01_raw_ingestion.py
# Purpose : Read pre-uploaded FHIR data, store raw JSON files bucketed by resource + date
# SERVERLESS 
# SETUP REQUIRED:
#   1. Download FHIR data on your local machine (with internet access)
#   2. Upload JSON files to: /Workspace/.../databricks-repo-FHIR/data/uploaded/
#   3. Expected files: patient_data.json, encounter_data.json, 
#                      observation_data.json, condition_data.json

import json, os
from pyspark.sql import SparkSession
import sys

# Add repository root to Python path for module imports
repo_root = "/Workspace/Users/vyshnavi.thunuguntla@outlook.com/databricks-repo-FHIR"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
    
from config.config import (
    FHIR_BASE_URL, RESOURCES, START_DATE, END_DATE,
    RAW_PATH, RESPONSE_FORMAT, PAGE_COUNT
)

spark = SparkSession.builder.getOrCreate()

# Path where pre-downloaded FHIR JSON files are uploaded
UPLOADED_DATA_PATH = f"{repo_root}/data/uploaded"


def ingest_resource_from_file(resource: str):
    """
    Read FHIR bundle from uploaded file instead of API call.
    Expected file format: {resource_lower}_data.json containing a FHIR bundle.
    """
    # Map resource name to expected filename
    input_file = f"{UPLOADED_DATA_PATH}/{resource.lower()}_data.json"
    
    # Check if file exists
    if not os.path.exists(input_file):
        raise FileNotFoundError(
            f"❌ Missing file: {input_file}\n"
            f"   Please upload {resource.lower()}_data.json to {UPLOADED_DATA_PATH}/\n"
            f"   See instructions in notebook header."
        )
    
    # Read the uploaded FHIR bundle
    print(f"📁 Reading {resource} data from: {input_file}")
    with open(input_file, "r") as f:
        bundle = json.load(f)
    
    # Validate it's a FHIR bundle
    if "resourceType" not in bundle or bundle["resourceType"] != "Bundle":
        raise ValueError(
            f"❌ Invalid FHIR bundle in {input_file}. "
            f"Expected resourceType='Bundle', got: {bundle.get('resourceType', 'missing')}"
        )
    
    # Wrap in list to match original "all_pages" structure
    all_pages = [bundle]
    entry_count = len(bundle.get("entry", []))
    
    # Write raw bundle JSON per resource per date
    out_dir = f"{RAW_PATH}/{resource}/date={END_DATE}"
    os.makedirs(out_dir, exist_ok=True)
    out_path = f"{out_dir}/bundle.json"
    
    with open(out_path, "w") as f:
        json.dump(all_pages, f, indent=2)
    
    print(f"[RAW] {resource}: {entry_count} entries, 1 bundle saved → {out_path}")
    return all_pages, input_file


# ── Run for all resources ─────────────────────────────────────────────────────
print(f"📂 Looking for uploaded files in: {UPLOADED_DATA_PATH}")
print(f"📅 Processing data for date: {END_DATE}\n")

raw_cache = {}
missing_files = []

for res in RESOURCES:
    try:
        pages, source_file = ingest_resource_from_file(res)
        raw_cache[res] = {"pages": pages, "source": source_file}
    except FileNotFoundError as e:
        print(str(e))
        missing_files.append(res)
    except Exception as e:
        print(f"❌ Error processing {res}: {str(e)}")
        missing_files.append(res)

if missing_files:
    print(f"\n⚠️  WARNING: {len(missing_files)} resource(s) failed: {missing_files}")
    print(f"\n💡 To fix: Upload the missing JSON files to {UPLOADED_DATA_PATH}/")
    print(f"   Expected filenames: {[f'{r.lower()}_data.json' for r in missing_files]}")
else:
    print("\n✅ Raw ingestion complete - all resources processed successfully!")

print(f"\n📊 Successfully processed: {len(raw_cache)}/{len(RESOURCES)} resources")